In [1]:
# ============================================
# TF-IDF Based Recommendation/Search System
# ============================================

# Import Libraries
import pandas as pd
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download required NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mohit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mohit\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:

# ============================================
# STEP 1: CREATE DATASET
# ============================================

documents = [
    {
        "title": "Cyber City",
        "description": "Hackers navigate a dystopian futuristic city controlled by a massive AI network."
    },
    {
        "title": "Mindful Meditation",
        "description": "A beginner's guide to reducing stress, staying present, and finding inner peace through meditation."
    },
    {
        "title": "Mountain Peaks",
        "description": "An incredible journey of climbers trying to conquer the world's tallest and deadliest mountains."
    },
    {
        "title": "Baking Basics",
        "description": "Learn the science behind baking bread, cakes, and pastries from scratch in your home kitchen."
    },
    {
        "title": "The Martian Colony",
        "description": "Humans establish their first permanent settlement on the red planet and face unexpected challenges."
    },
    {
        "title": "Personal Finance 101",
        "description": "Essential tips for budgeting, investing, and achieving long-term financial independence."
    },
    {
        "title": "Quantum Computing Explained",
        "description": "A deep dive into how quantum computers work and how they will change the future of technology."
    },
    {
        "title": "Wildlife Safari",
        "description": "Photographers travel through the African savanna to capture images of rare and endangered animals."
    },
    {
        "title": "The Dragon's Lair",
        "description": "A group of brave knights travels across the kingdom to defeat a mysterious fire-breathing dragon."
    },
    {
        "title": "Urban Gardening",
        "description": "Tips and tricks for growing your own vegetables and herbs in small apartment spaces."
    }
]

# Convert to DataFrame
df = pd.DataFrame(documents)

print("DATASET")
print(df)

DATASET
                         title  \
0                   Cyber City   
1           Mindful Meditation   
2               Mountain Peaks   
3                Baking Basics   
4           The Martian Colony   
5         Personal Finance 101   
6  Quantum Computing Explained   
7              Wildlife Safari   
8            The Dragon's Lair   
9              Urban Gardening   

                                         description  
0  Hackers navigate a dystopian futuristic city c...  
1  A beginner's guide to reducing stress, staying...  
2  An incredible journey of climbers trying to co...  
3  Learn the science behind baking bread, cakes, ...  
4  Humans establish their first permanent settlem...  
5  Essential tips for budgeting, investing, and a...  
6  A deep dive into how quantum computers work an...  
7  Photographers travel through the African savan...  
8  A group of brave knights travels across the ki...  
9  Tips and tricks for growing your own vegetable...  


In [3]:
# ============================================
# STEP 2: TEXT PREPROCESSING
# ============================================

# Initialize tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):

    # Lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Tokenization
    words = text.split()

    # Remove stopwords + Lemmatization
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    # Join words back
    return " ".join(words)

# Apply preprocessing
df['processed_text'] = (
    df['title'] + " " + df['description']
).apply(preprocess_text)

print("\nPROCESSED TEXT")
print(df[['title', 'processed_text']])



PROCESSED TEXT
                         title  \
0                   Cyber City   
1           Mindful Meditation   
2               Mountain Peaks   
3                Baking Basics   
4           The Martian Colony   
5         Personal Finance 101   
6  Quantum Computing Explained   
7              Wildlife Safari   
8            The Dragon's Lair   
9              Urban Gardening   

                                      processed_text  
0  cyber city hacker navigate dystopian futuristi...  
1  mindful meditation beginner guide reducing str...  
2  mountain peak incredible journey climber tryin...  
3  baking basic learn science behind baking bread...  
4  martian colony human establish first permanent...  
5  personal finance 101 essential tip budgeting i...  
6  quantum computing explained deep dive quantum ...  
7  wildlife safari photographer travel african sa...  
8  dragon lair group brave knight travel across k...  
9  urban gardening tip trick growing vegetable he...  


In [4]:
# ============================================
# STEP 3: TF-IDF VECTORIZATION
# ============================================

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(df['processed_text'])

print("\nTF-IDF MATRIX SHAPE")
print(tfidf_matrix.shape)



TF-IDF MATRIX SHAPE
(10, 105)


In [5]:



# ============================================
# STEP 4: USER QUERY
# ============================================

user_query = "advanced computer technology and AI"

# Preprocess query
processed_query = preprocess_text(user_query)

print("\nUSER QUERY")
print(user_query)

print("\nPROCESSED QUERY")
print(processed_query)

# Convert query to TF-IDF vector
query_vector = vectorizer.transform([processed_query])



USER QUERY
advanced computer technology and AI

PROCESSED QUERY
advanced computer technology ai


In [6]:

# ============================================
# STEP 5: COSINE SIMILARITY
# ============================================

similarity_scores = cosine_similarity(query_vector, tfidf_matrix)

# Convert scores to list
scores = similarity_scores[0]

# Add scores to dataframe
df['similarity_score'] = scores


In [7]:

# ============================================
# STEP 6: TOP 3 RECOMMENDATIONS
# ============================================

top_results = df.sort_values(
    by='similarity_score',
    ascending=False
).head(3)

print("\nTOP 3 RECOMMENDATIONS\n")

for index, row in top_results.iterrows():

    print("Title:", row['title'])
    print("Description:", row['description'])
    print("Similarity Score:", round(row['similarity_score'], 4))
    print("-" * 50)


TOP 3 RECOMMENDATIONS

Title: Quantum Computing Explained
Description: A deep dive into how quantum computers work and how they will change the future of technology.
Similarity Score: 0.3203
--------------------------------------------------
Title: Cyber City
Description: Hackers navigate a dystopian futuristic city controlled by a massive AI network.
Similarity Score: 0.1601
--------------------------------------------------
Title: Mindful Meditation
Description: A beginner's guide to reducing stress, staying present, and finding inner peace through meditation.
Similarity Score: 0.0
--------------------------------------------------
